# IMDb Movie Genre Prediction: Final Project Report
Autors: Szymon Pająk, Modar Suliman, Przemysław Maresz

## 1. Introduction & Project Overview

### Objective
The primary goal of this project is to develop a robust machine learning pipeline capable of predicting movie genres given their textual metadata. Accurately tagging movies with genres is critical for recommendation systems, search optimization, and content organization. 

This report documents the end-to-end process, from data acquisition and exploratory analysis to the implementation and evaluation of advanced Natural Language Processing (NLP) models. We specifically compare traditional approaches (TF-IDF) with modern deep learning representations (SBERT).

## 2. Data Acquisition Pipeline

The dataset was curated from multiple sources to ensure richness and reliability.

### 2.1 Core Metadata (IMDb Datasets)
We utilized the official IMDb non-commercial datasets:
-   **title.basics.tsv.gz:** Provided the unique identifiers (`tconst`), titles, release years, and genres.
-   **title.akas.tsv.gz & title.ratings.tsv.gz:** Used to filter for region relevance and popularity (vote counts).

### 2.2 Enrichment (Web Scraping)
The core dataset lacked detailed textual descriptions. We implemented a scraping module to fetch enriched data for each movie from its IMDb page:
-   **Plot Descriptions:** Long-form text describing the movie's narrative.
-   **AI Summaries:** Concise, generated summaries available on newer IMDb pages.
-   **Keywords:** Semantic tags (e.g., "car chase", "friendship") that provide strong signal for classification.

### 2.3 Data Preparation
To create a clean learning environment, we filtered the data with the following criteria:
-   **Scope:** Released in or after **1980**.
-   **Type:** `movie` or `tvMovie`.
-   **Popularity:** Minimum **1,000 votes** (to remove obscure outliers).
-   **Region:** English-speaking or Western markets (US, UK, CA, etc.).

**Feature Construction:**
We aggregated the textual signals into a single input feature for the models:
> `Text = Description + " " + Summary + " " + Keywords`

## 3. Exploratory Data Analysis (EDA)

### 3.1 Genre Distribution
An analysis of the target labels revealed a classic "long tail" distribution. A few genres dominate the industry, while many others appear infrequently.

**Distribution of Top Genres:**

| Genre | Frequency (Count) |
| :--- | :---: |
| **Drama** | 5,589 |
| **Comedy** | 3,648 |
| **Action** | 2,605 |
| Crime | 2,103 |
| Adventure | 1,726 |
| Thriller | 1,711 |
| Romance | 1,643 |
| Horror | 1,309 |
| Mystery | 1,162 |
| Sci-Fi | 700 |
| ... | ... |
| Western | 49 |

### 3.2 Imbalance & Simplification Strategy
Predicting 23+ classes with such extreme imbalance is challenging. We hypothesized that focusing on the most prevalent descriptors would yield a more useful model.

**Coverage Analysis:**
We checked how many movies contain at least one of the "Big Three" genres: *Drama*, *Comedy*, or *Action*.

| Segment | Movie Count | Percentage |
| :--- | :---: | :---: |
| **Contains Top 3 Genre** | **8,947** | **89.5%** |
| No Top 3 Genre | 1,053 | 10.5% |

**Decision:** We simplified the multi-label problem into **4 categories**:
1.  Drama
2.  Comedy
3.  Action
4.  Other (Any movie not fitting the above)

## 4. Methodology

### 4.1 Feature Extraction Approaches
We compared two fundamental approaches to representing text as numbers:
-   **TF-IDF (Term Frequency-Inverse Document Frequency):** A statistical measure that evaluates how relevant a word is to a document in a collection. It is a sparse, high-dimensional representation.
-   **SBERT (Sentence-BERT):** A modification of the BERT network that uses siamese networks to derive semantically meaningful sentence embeddings. We used `sentence-transformers/all-MiniLM-L6-v2`. This produces dense, 384-dimensional vectors that capture context/meaning rather than just keyword presence.

### 4.2 Classification Models
-   **Logistic Regression (One-vs-Rest):** Serves as a strong linear baseline.
-   **LightGBM:** A gradient boosting framework that uses tree-based learning algorithms. Known for speed and accuracy on dense features.
-   **MLP (Multi-Layer Perceptron):** A deep neural network built with Keras/TensorFlow to capture non-linear dependencies.
-   **Classifier Chains:** An ensemble method that models the correlation between labels (e.g., *Action* and *Adventure* often appear together) by passing the output of one classifier as input to the next.

## 5. Phase 1: Benchmark Experiments (Original 23 Classes)

We first attempted to predict the original raw genres to establish a baseline. This phase highlighted the difficulty of the task before dataset simplification.

### Phase 1 Results

| Model | Feature Type | Micro F1 | Macro F1 | Micro Precision | Micro Recall |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Logistic Regression** | **TF-IDF** | 0.4772 | 0.1794 | 0.7261 | 0.3554 |
| **LightGBM (OVR)** | **SBERT** | **0.6119** | **0.4140** | **0.5888** | 0.6369 |
| **MLP (Dense Net)** | **SBERT** | 0.5890 | 0.4452 | 0.5104 | **0.6961** |
| **Classifier Chains** | **SBERT** | 0.5791 | 0.4219 | 0.5150 | 0.6614 |

**Analysis:**
-   **The Failure of TF-IDF:** While TF-IDF achieved high precision (0.72), its recall was abysmal (0.35). It could identify obvious keywords but completely missed subtle genre indicators.
-   **SBERT Superiority:** Switching to SBERT embeddings immediately jumped the F1 score from ~0.48 to ~0.58-0.61. The embeddings captured the *semantic* meaning of the plots.

## 6. Phase 2: Final Results (Improved 4-Class Architecture)

Applying the 4-class simplification strategy alongside SBERT embeddings yielded our best results. The models utilized `OneVsRest` strategies or dedicated multi-label loss functions (`BinaryFocalCrossentropy`).

### Phase 2 Performance Comparison

| Model | Micro F1 | Macro F1 | Micro Precision | Micro Recall |
| :--- | :---: | :---: | :---: | :---: |
| **SBERT + Logistic Regression** | **0.8256** | 0.7745 | **0.8255** | 0.8257 |
| **SBERT + LightGBM (OVR)** | 0.8251 | **0.7828** | 0.7781 | 0.8783 |
| **SBERT + MLP (Dense Net)** | 0.8233 | 0.7819 | 0.7660 | **0.8897** |
| **SBERT + Classifier Chains** | 0.8190 | 0.7781 | 0.7906 | 0.8495 |

### Deep Dive: Per-Class Metrics
To understand *where* the models excel, we analyze the breakdown for the top performers.

**1. Logistic Regression (Most Balanced)**
Offers the best trade-off between Precision and Recall.

| Class | Precision | Recall | F1-Score | Support |
| :--- | :---: | :---: | :---: | :---: |
| **Action** | 0.76 | 0.65 | 0.70 | 543 |
| **Comedy** | 0.75 | 0.62 | 0.68 | 722 |
| **Drama** | 0.76 | 0.79 | 0.77 | 1142 |
| **Other** | 0.90 | 0.98 | 0.94 | 1792 |

**2. LightGBM (Highest Recall)**
If the goal is to never miss a genre tag (even if it means occasionally over-tagging), LightGBM is superior.

| Class | Precision | Recall | F1-Score | Support |
| :--- | :---: | :---: | :---: | :---: |
| **Action** | 0.71 | 0.74 | 0.73 | 543 |
| **Comedy** | 0.62 | 0.76 | 0.68 | 722 |
| **Drama** | 0.73 | 0.83 | 0.78 | 1142 |
| **Other** | 0.90 | 1.00 | 0.95 | 1792 |

## 7. Discussion & Conclusion

This analytics pipeline successfully demonstrates that high-quality genre prediction is achievable through a combination of modern NLP and domain-aware data preprocessing.

**Key Findings:**
1.  **Embeddings Matter:** SBERT provides a massive lift over frequency-based vectors (TF-IDF), proving that "understanding" the plot summary is better than just counting words.
2.  **Recall vs. Precision:** Deep learning models (MLP) and Gradient Boosting (LightGBM) are significantly better at Recall (~88%) compared to linear models. They are "braver" at assigning tags.
3.  **Data Strategy:** The simplification from 23 noisy classes to 4 distinct categories was the single most effective optimization step, raising the F1 score ceiling from 0.61 to 0.83.